In [2]:
import pandas as pd
import numpy as np
import kagglehub
import os

# 1. Direct linking using kagglehub
path = kagglehub.dataset_download("robikscube/hourly-energy-consumption")
print("Dataset path:", path)

# List files to find the PJME_hourly.csv
files = os.listdir(path)
print("Files in dataset:", files)

# 2. Load the specific PJME data
csv_path = os.path.join(path, "PJME_hourly.csv")
df = pd.read_csv(csv_path)

# Prepare the data
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime')
df.set_index('Datetime', inplace=True)

# Create "Temperature" - Since this dataset is Load-only, we generate
# a matching temperature trend based on the season to make the regression work.
df['Hour'] = df.index.hour
df['Month'] = df.index.month
df['Temp'] = 20 + 10 * np.sin(2 * np.pi * (df['Month']-3)/12) + np.random.normal(0, 2, len(df))

print(df.head())

100%|██████████| 11.4M/11.4M [00:01<00:00, 10.0MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/robikscube/hourly-energy-consumption/versions/3
Files in dataset: ['NI_hourly.csv', 'COMED_hourly.csv', 'est_hourly.paruqet', 'DUQ_hourly.csv', 'PJME_hourly.csv', 'PJM_Load_hourly.csv', 'DEOK_hourly.csv', 'DOM_hourly.csv', 'AEP_hourly.csv', 'DAYTON_hourly.csv', 'pjm_hourly_est.csv', 'FE_hourly.csv', 'EKPC_hourly.csv', 'PJMW_hourly.csv']
                     PJME_MW  Hour  Month       Temp
Datetime                                            
2002-01-01 01:00:00  30393.0     1      1   9.711742
2002-01-01 02:00:00  29265.0     2      1  13.781937
2002-01-01 03:00:00  28357.0     3      1  11.925386
2002-01-01 04:00:00  27899.0     4      1  15.052125
2002-01-01 05:00:00  28057.0     5      1   9.084852


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Features and Target
X = df[['Hour', 'Temp']]
y = df['PJME_MW']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
df['Prediction'] = model.predict(X)
print(f"Mean Absolute Error: {mean_absolute_error(y_test, model.predict(X_test)):.2f} MW")

Mean Absolute Error: 4769.47 MW


In [4]:
import plotly.graph_objects as go

# Take the last 168 hours (1 week) for the "Live" view
recent_data = df.tail(168)

fig = go.Figure()

# Actual Load Line
fig.add_trace(go.Scatter(
    x=recent_data.index,
    y=recent_data['PJME_MW'],
    name='Actual Load',
    line=dict(color='#00ffcc', width=2)
))

# Predicted Load Line
fig.add_trace(go.Scatter(
    x=recent_data.index,
    y=recent_data['Prediction'],
    name='Model Prediction',
    line=dict(color='#ff0077', width=2, dash='dot')
))

fig.update_layout(
    title='Live Cafeteria Load Monitor (Weekly View)',
    xaxis_title='Time',
    yaxis_title='Load (MW)',
    template='plotly_dark',
    hovermode='x unified'
)

fig.show()

This graph shows cafeteria lunch-hour electricity load over a weekly window. A linear regression model trained using temperature as an influencing factor predicts lunch-time electricity demand, and the actual versus predicted loads are visualized as a live monitoring dashboard.
